# Demo — Trazabilidad ETL: Bronze → Silver → Gold

Tomamos **registros concretos** de tres fuentes distintas y rastreamos campo a campo qué le pasa a cada valor
en cada capa de transformación.

| Capa | Propósito | Tecnología |
|------|-----------|------------|
| **Bronze** | Ingestión cruda — datos tal como llegan de la fuente | Databricks Delta Lake · `sandbox_bronze_ss2` |
| **Silver** | Limpieza, tipado y normalización | Databricks Delta Lake · `stage_silver_ss2` |
| **Gold** | Modelo dimensional Galaxy Schema | Databricks + PostgreSQL local · `gold_ss2` |

> **Nota:** Bronze y Silver viven en Databricks (no en el PostgreSQL local). Los registros de esas capas
> se muestran tal como quedaron guardados en Delta Lake; Gold se consulta en vivo desde PostgreSQL.

In [ ]:
import os
from pathlib import Path
import psycopg
import pandas as pd
from IPython.display import display, HTML
from dotenv import load_dotenv

load_dotenv(Path('.env'))

CONN = dict(
    host='localhost',
    port=int(os.environ['PG_PORT']),
    dbname=os.environ['PG_DB'],
    user=os.environ['PG_USER'],
    password=os.environ['PG_PASSWORD'],
)

conn = psycopg.connect(**CONN)
print('Conexión al Gold (PostgreSQL) OK —', conn.info.server_version)

def query(sql: str) -> pd.DataFrame:
    with conn.cursor() as cur:
        cur.execute(sql)
        cols = [desc[0] for desc in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

In [ ]:
display(HTML("""
<style>
.jp-OutputArea-output, .output_area, .output_subarea {
    background: #0d1117 !important;
}
.jp-OutputArea-output pre, .output_text pre, .output_stream pre {
    color: #7ee787 !important;
    background: #161b22 !important;
    padding: 8px 14px !important;
    border-radius: 6px !important;
    border-left: 3px solid #238636 !important;
    font-family: 'Cascadia Code', 'Fira Code', monospace !important;
}
div.jp-MarkdownOutput { background: #0d1117 !important; padding: 4px 12px !important; border-radius: 6px !important; }
div.jp-MarkdownOutput h1 { color: #79c0ff !important; border-bottom: 1px solid #21262d !important; }
div.jp-MarkdownOutput h2 { color: #79c0ff !important; }
div.jp-MarkdownOutput h3 { color: #79c0ff !important; }
div.jp-MarkdownOutput p, div.jp-MarkdownOutput li { color: #c9d1d9 !important; }
div.jp-MarkdownOutput strong { color: #f0f6fc !important; }
div.jp-MarkdownOutput table { border-color: #30363d !important; width: 100%; }
div.jp-MarkdownOutput th { background: #21262d !important; color: #79c0ff !important; border-color: #30363d !important; padding: 8px 12px !important; }
div.jp-MarkdownOutput td { background: #161b22 !important; color: #c9d1d9 !important; border-color: #30363d !important; padding: 6px 12px !important; }
div.jp-MarkdownOutput code { background: #1f2937 !important; color: #e3b341 !important; border-radius: 3px !important; padding: 1px 6px !important; }
div.jp-MarkdownOutput pre code { background: #161b22 !important; color: #c9d1d9 !important; padding: 12px !important; display: block !important; border-radius: 6px !important; }
div.jp-MarkdownOutput blockquote { border-left: 3px solid #30363d !important; color: #8b949e !important; margin-left: 0 !important; padding-left: 12px !important; }
div.jp-MarkdownOutput hr { border-color: #21262d !important; }
</style>
"""))

In [ ]:
def mostrar_evolucion(registros: dict, titulo: str = ""):
    bronze = registros.get('Bronze', {})
    silver = registros.get('Silver', {})
    gold   = registros.get('Gold',   {})

    all_fields = list(bronze.keys())
    for k in silver:
        if k not in all_fields:
            all_fields.append(k)
    for k in gold:
        if k not in all_fields:
            all_fields.append(k)

    DARK_BG     = '#0d1117'
    CARD_BG     = '#161b22'
    BORDER      = '#30363d'
    TEXT        = '#c9d1d9'
    MUTED       = '#484f58'
    CHANGED_BG  = '#2d2400'
    CHANGED_TXT = '#e3b341'
    NEW_BG      = '#0d2818'
    NEW_TXT     = '#56d364'
    RAW_BG      = '#1c2433'
    RAW_TXT     = '#58a6ff'

    def estilo(layer_dict, field, prev_dict):
        if field not in layer_dict:
            return f'background:{CARD_BG};color:{MUTED}'
        if field not in prev_dict:
            return f'background:{NEW_BG};color:{NEW_TXT}'
        if str(layer_dict[field]) != str(prev_dict.get(field, '')):
            return f'background:{CHANGED_BG};color:{CHANGED_TXT}'
        return f'background:{CARD_BG};color:{TEXT}'

    def fmt(v):
        if v == '—':
            return f'<span style="color:{MUTED}">—</span>'
        return f'<span>{v}</span>'

    rows = []
    for field in all_fields:
        b_val = bronze.get(field, '—')
        s_val = silver.get(field, '—')
        g_val = gold.get(field,   '—')
        b_bg  = f'background:{RAW_BG};color:{RAW_TXT}' if field in bronze else f'background:{CARD_BG};color:{MUTED}'
        s_bg  = estilo(silver, field, bronze)
        g_bg  = estilo(gold,   field, silver)
        rows.append(
            f'<tr>'
            f'<td style="font-weight:600;padding:7px 14px;border:1px solid {BORDER};font-size:13px;white-space:nowrap;color:{TEXT};background:{CARD_BG}">{field}</td>'
            f'<td style="{b_bg};padding:7px 14px;border:1px solid {BORDER};font-family:monospace;font-size:12px">{fmt(b_val)}</td>'
            f'<td style="{s_bg};padding:7px 14px;border:1px solid {BORDER};font-family:monospace;font-size:12px">{fmt(s_val)}</td>'
            f'<td style="{g_bg};padding:7px 14px;border:1px solid {BORDER};font-family:monospace;font-size:12px">{fmt(g_val)}</td>'
            f'</tr>'
        )

    legend = (
        f'<div style="margin:10px 0 16px;font-size:12px;display:flex;gap:12px;flex-wrap:wrap">'
        f'<span style="background:{NEW_BG};color:{NEW_TXT};padding:3px 10px;border-radius:4px;border:1px solid #238636">Campo nuevo en esta capa</span>'
        f'<span style="background:{CHANGED_BG};color:{CHANGED_TXT};padding:3px 10px;border-radius:4px;border:1px solid #9e6a03">Valor transformado</span>'
        f'<span style="background:{RAW_BG};color:{RAW_TXT};padding:3px 10px;border-radius:4px;border:1px solid #1f6feb">Dato crudo sin cambio</span>'
        f'</div>'
    )
    header = (
        f'<tr>'
        f'<th style="background:#21262d;color:#79c0ff;padding:9px 14px;border:1px solid {BORDER};text-align:left;font-size:13px">Campo</th>'
        f'<th style="background:#3d2012;color:#ffa657;padding:9px 14px;border:1px solid {BORDER};text-align:center">BRONZE<br><small style="font-weight:normal;color:#d2a679">datos crudos</small></th>'
        f'<th style="background:#112233;color:#79c0ff;padding:9px 14px;border:1px solid {BORDER};text-align:center">SILVER<br><small style="font-weight:normal;color:#8ab4d4">limpieza y normalización</small></th>'
        f'<th style="background:#2d1f00;color:#f0c040;padding:9px 14px;border:1px solid {BORDER};text-align:center">GOLD<br><small style="font-weight:normal;color:#c9a030">modelo dimensional</small></th>'
        f'</tr>'
    )
    html = (
        f'<div style="font-family:-apple-system,BlinkMacSystemFont,\'Segoe UI\',sans-serif;margin:24px 0;'
        f'background:{DARK_BG};padding:20px 20px 16px;border-radius:10px;border:1px solid {BORDER}">'
        f'<h3 style="color:#79c0ff;border-left:4px solid #1f6feb;padding-left:12px;margin:0 0 12px;font-size:15px;font-weight:600">{titulo}</h3>'
        f'{legend}'
        f'<div style="overflow-x:auto;border-radius:6px;border:1px solid {BORDER}">'
        f'<table style="border-collapse:collapse;min-width:700px;width:100%;background:{CARD_BG}">'
        f'<thead>{header}</thead>'
        f'<tbody>{chr(10).join(rows)}</tbody>'
        f'</table></div></div>'
    )
    display(HTML(html))


def dark_df(df: pd.DataFrame) -> object:
    return df.style.set_properties(**{
        'background-color': '#161b22',
        'color': '#c9d1d9',
        'border-color': '#30363d',
        'font-size': '12px',
        'padding': '6px 12px',
    }).set_table_styles([
        {'selector': 'thead th', 'props': [
            ('background-color', '#21262d'),
            ('color', '#79c0ff'),
            ('border-bottom', '2px solid #30363d'),
            ('padding', '8px 12px'),
            ('font-size', '12px'),
            ('text-align', 'left'),
        ]},
        {'selector': 'tbody tr:nth-child(even) td', 'props': [
            ('background-color', '#1a1f27'),
        ]},
        {'selector': 'tbody tr:hover td', 'props': [
            ('background-color', '#21262d'),
        ]},
        {'selector': 'td', 'props': [
            ('border-bottom', '1px solid #21262d'),
        ]},
        {'selector': 'table', 'props': [
            ('border-radius', '6px'),
            ('border', '1px solid #30363d'),
            ('overflow', 'hidden'),
            ('width', '100%'),
        ]},
    ])

---
## Arquitectura del pipeline

```
Fuentes externas
  INE (S3)  ──────────────────────────────────────────────────────────┐
  MSPAS (Dropbox) ─────────────────────────────────────────────────── │
  WHO (Google Drive / Volume) ──────────────────────────────────────── │
                                                                       ▼
                                               ┌─────────────────────────────┐
                                               │  BRONZE (sandbox_bronze_ss2) │
                                               │  9 tablas Delta Lake         │
                                               │  + columnas de linaje        │
                                               └────────────┬────────────────┘
                                                            │  stage_*.ipynb
                                                            ▼
                                               ┌─────────────────────────────┐
                                               │  SILVER (stage_silver_ss2)  │
                                               │  9 tablas Delta Lake         │
                                               │  tipos correctos, CIE-10     │
                                               │  normalizado, sin subtotales │
                                               └────────────┬────────────────┘
                                                            │  gold_*.ipynb
                                                            ▼
                                 ┌──────────────────────────────────────────────┐
                                 │  GOLD (gold_ss2)  —  Galaxy Schema            │
                                 │  dim_tiempo  dim_geografia  dim_causa          │
                                 │  dim_demografia  dim_fuente                    │
                                 │  fact_defunciones  fact_morbilidad             │
                                 └────────────────────┬─────────────────────────┘
                                                      │  load-postgresql.py
                                                      ▼
                                         PostgreSQL local (este notebook)
```

---
## Ejemplo 1 — WHO Mortalidad: normalización de idioma, tipos y grupos etarios

Seguimos un registro de **muertes por COVID-19 en Guatemala, 2020, mujeres de 25-29 años**,
tal como entra desde el portal WHO Mortality Database y cómo queda en Gold.

In [ ]:
who_bronze = {
    'year':                   '2020',
    'country':                'guatemala',
    'Gender':                 'Female',
    'age_group_code':         'Age25_29',
    'indicator_code':         'SA.D.COVID19',
    'indicator_name':         'Coronavirus disease (COVID-19)',
    'number':                 '142',
    'crude_death_rate':       '0.82',
    'age_std_death_rate':     '0.79',
    'pct_cause_total_deaths': '1.45',
    'ingestion_timestamp':    '2024-11-20 08:30:00',
    'source_system':          'WHO Mortality Database Portal',
    'source_file':            'WHO_Guatemala_2020.csv',
}

who_silver = {
    'anio':                       2020,
    'pais':                       'Guatemala',
    'sexo':                       'F',
    'grupo_etario':               '25-29',
    'codigo_causa':               'SA.D.COVID19',
    'nombre_causa':               'Enfermedad por coronavirus (COVID-19)',
    'defunciones':                142,
    'tasa_mortalidad_x100k':      0.82,
    'tasa_estandarizada_x100k':   0.79,
    'pct_causa_total_muertes':    1.45,
    'silver_processed_timestamp': '2024-11-20 09:15:44',
    'silver_job_run_id':          'run-who-gt-001',
    'ingestion_timestamp':        '2024-11-20 08:30:00',
    'source_system':              'WHO Mortality Database Portal',
    'source_file':                'WHO_Guatemala_2020.csv',
}

who_gold = {
    'tiempo_sk':                9,
    'geografia_sk':             1,
    'demografia_sk':            15,
    'causa_sk':                 87,
    'fuente_sk':                3,
    'total_defunciones':        142,
    'defunciones_hombres':      None,
    'defunciones_mujeres':      None,
    'tasa_mortalidad_x100k':    0.82,
    'tasa_estandarizada_x100k': 0.79,
}

mostrar_evolucion(
    {'Bronze': who_bronze, 'Silver': who_silver, 'Gold': who_gold},
    titulo='Ejemplo 1 — WHO: defunción COVID-19, Guatemala 2020, Mujeres 25-29'
)

In [ ]:
SQL_WHO_GOLD = """
SELECT
    t.anio, t.periodo_covid,
    g.pais, g.nivel_geo,
    d.sexo, d.grupo_etario_label,
    c.codigo_origen, c.descripcion,
    fu.source_system,
    f.total_defunciones, f.tasa_mortalidad_x100k, f.tasa_estandarizada_x100k
FROM gold_ss2.fact_defunciones f
JOIN gold_ss2.dim_tiempo     t  ON f.tiempo_sk    = t.tiempo_sk
JOIN gold_ss2.dim_geografia  g  ON f.geografia_sk = g.geografia_sk
JOIN gold_ss2.dim_demografia d  ON f.demografia_sk = d.demografia_sk
JOIN gold_ss2.dim_causa      c  ON f.causa_sk     = c.causa_sk
JOIN gold_ss2.dim_fuente     fu ON f.fuente_sk    = fu.fuente_sk
WHERE t.anio = 2020 AND g.pais = 'Guatemala' AND g.nivel_geo = 'pais'
  AND d.sexo = 'F' AND fu.source_system ILIKE '%WHO%'
ORDER BY f.total_defunciones DESC
LIMIT 8
"""

df_who_gold = query(SQL_WHO_GOLD)
print(f'Registros en Gold (Guatemala 2020, mujeres, fuente WHO): {len(df_who_gold)}')
dark_df(df_who_gold)

---
## Ejemplo 2 — INE Defunciones: normalización CIE-10 y filtrado de subtotales

El INE entrega los códigos CIE-10 con separadores irregulares (`J:18:0`).
Silver los limpia a formato estándar (`J180`). Además los archivos INE incluyen
filas de subtotal (`'Todas las causas'`) que se **eliminan** en Silver.

In [ ]:
ine_bronze = {
    'anio':            '2020',
    'sexo':            'Hombres',
    'edad':            '25-29',
    'codigo_cie_10':   'J:18:0',
    'causa_de_muerte': 'Neumonía, no especificada',
    'total':           '89',
    'hombres':         '89',
    'mujeres':         '0',
    'ingestion_timestamp': '2024-11-20 07:55:12',
    'source_system':       'INE_DEFUNCIONES',
    'source_file':         's3://bucket-ss2/ine/defunciones_sexo_edad_2020.xlsx',
}

ine_bronze_subtotal = {
    'anio':            '2020',
    'sexo':            'Total',
    'edad':            'Todas las edades',
    'codigo_cie_10':   'Todas las causas',
    'causa_de_muerte': 'Todas las causas',
    'total':           '42115',
    'hombres':         '22890',
    'mujeres':         '19225',
    'ingestion_timestamp': '2024-11-20 07:55:12',
    'source_system':       'INE_DEFUNCIONES',
    'source_file':         's3://bucket-ss2/ine/defunciones_sexo_edad_2020.xlsx',
}

ine_silver = {
    'anio':            2020,
    'sexo':            'Hombres',
    'edad':            '25-29',
    'cie_10_norm':     'J180',
    'codigo_cie_10':   'J:18:0',
    'causa_de_muerte': 'Neumonía, no especificada',
    'total':           89,
    'hombres':         89,
    'mujeres':         0,
    'silver_processed_timestamp': '2024-11-20 09:20:05',
    'silver_job_run_id':          'run-ine-edad-001',
    'ingestion_timestamp': '2024-11-20 07:55:12',
    'source_system':       'INE_DEFUNCIONES',
    'source_file':         's3://bucket-ss2/ine/defunciones_sexo_edad_2020.xlsx',
}

ine_gold = {
    'tiempo_sk':                9,
    'geografia_sk':             2,
    'demografia_sk':            22,
    'causa_sk':                 134,
    'fuente_sk':                1,
    'total_defunciones':        89,
    'defunciones_hombres':      89,
    'defunciones_mujeres':      0,
    'tasa_mortalidad_x100k':    None,
    'tasa_estandarizada_x100k': None,
}

mostrar_evolucion(
    {'Bronze': ine_bronze, 'Silver': ine_silver, 'Gold': ine_gold},
    titulo='Ejemplo 2 — INE: defunción por neumonía (J18.0), Hombres 25-29, Guatemala 2020'
)

display(HTML(
    '<div style="background:#1a0a0a;border-left:4px solid #e53935;padding:10px 16px;'
    'border-radius:4px;font-family:sans-serif;font-size:13px;margin:10px 0;color:#f48771">'
    '<strong>Fila de subtotal eliminada en Silver:</strong><br>'
    '<code style="color:#e3b341">anio=2020 | sexo=Total | edad=Todas las edades | codigo_cie_10=\'Todas las causas\' | total=42115</code><br>'
    '<span style="color:#c9d1d9">Silver filtra las filas donde <code style="color:#e3b341">causa_de_muerte IN (\'Todas las causas\', \'Otras causas\')</code></span>'
    '</div>'
))

In [ ]:
SQL_INE_GOLD = """
SELECT
    t.anio, t.periodo_covid,
    g.pais, g.departamento,
    d.sexo, d.grupo_etario_label,
    c.codigo_origen, c.descripcion,
    fu.pipeline_name,
    f.total_defunciones, f.defunciones_hombres, f.defunciones_mujeres
FROM gold_ss2.fact_defunciones f
JOIN gold_ss2.dim_tiempo     t  ON f.tiempo_sk    = t.tiempo_sk
JOIN gold_ss2.dim_geografia  g  ON f.geografia_sk = g.geografia_sk
JOIN gold_ss2.dim_demografia d  ON f.demografia_sk = d.demografia_sk
JOIN gold_ss2.dim_causa      c  ON f.causa_sk     = c.causa_sk
JOIN gold_ss2.dim_fuente     fu ON f.fuente_sk    = fu.fuente_sk
WHERE t.anio = 2020 AND c.codigo_origen LIKE 'J18%'
  AND fu.pipeline_name ILIKE '%ine%edad%'
ORDER BY d.sexo, d.grupo_etario_label
LIMIT 10
"""

df_ine_gold = query(SQL_INE_GOLD)
print(f'Registros en Gold (código J18x, INE-edad, 2020): {len(df_ine_gold)}')
dark_df(df_ine_gold)

---
## Ejemplo 3 — MSPAS Morbilidad: COALESCE de columnas y resolución de departamento

Los archivos MSPAS tienen dos columnas para casos según el período (`casos` / `cantidad`).
Silver los unifica con `COALESCE`. Además el nombre del departamento llega en mayúsculas
y se resuelve contra el catálogo oficial con tildes y casing correcto.

In [ ]:
mspas_bronze = {
    'anio':                2020.0,
    'departamento':        'CIUDAD DE GUATEMALA',
    'municipio':           'Zona 1',
    'codigo_cie_10':       'J:06',
    'causa_de_morbilidad': 'Infecciones agudas de las vías respiratorias superiores',
    'casos':               None,
    'cantidad':            1840,
    'sexo':                'F',
    'ingestion_timestamp': '2024-11-20 08:10:33',
    'source_system':       'MSPAS',
    'source_file':         'primeras_causas_morbilidad_2020.csv',
}

mspas_silver = {
    'anio':                       2020,
    'departamento_oficial':       'Guatemala',
    'departamento':               'CIUDAD DE GUATEMALA',
    'municipio':                  'Zona 1',
    'cie_10_norm':                'J06',
    'codigo_cie_10':              'J:06',
    'causa_de_morbilidad':        'Infecciones agudas de las vías respiratorias superiores',
    'casos_total':                1840,
    'casos':                      None,
    'cantidad':                   1840,
    'sexo':                       'F',
    'silver_processed_timestamp': '2024-11-20 09:30:18',
    'silver_job_run_id':          'run-mspas-001',
    'ingestion_timestamp':        '2024-11-20 08:10:33',
    'source_system':              'MSPAS',
    'source_file':                'primeras_causas_morbilidad_2020.csv',
}

mspas_gold = {
    'tiempo_sk':    9,
    'geografia_sk': 48,
    'demografia_sk':10,
    'causa_sk':     55,
    'fuente_sk':    6,
    'casos_total':  1840,
}

mostrar_evolucion(
    {'Bronze': mspas_bronze, 'Silver': mspas_silver, 'Gold': mspas_gold},
    titulo='Ejemplo 3 — MSPAS: morbilidad respiratoria (J06), Guatemala Zona 1, 2020, Mujeres'
)

In [ ]:
SQL_MSPAS_GOLD = """
SELECT
    t.anio, t.periodo_covid,
    g.departamento, g.municipio, g.nivel_geo,
    d.sexo,
    c.codigo_origen, c.descripcion,
    fu.pipeline_name,
    f.casos_total
FROM gold_ss2.fact_morbilidad f
JOIN gold_ss2.dim_tiempo     t  ON f.tiempo_sk    = t.tiempo_sk
JOIN gold_ss2.dim_geografia  g  ON f.geografia_sk = g.geografia_sk
JOIN gold_ss2.dim_demografia d  ON f.demografia_sk = d.demografia_sk
JOIN gold_ss2.dim_causa      c  ON f.causa_sk     = c.causa_sk
JOIN gold_ss2.dim_fuente     fu ON f.fuente_sk    = fu.fuente_sk
WHERE t.anio = 2020 AND g.departamento = 'Guatemala'
  AND c.codigo_origen LIKE 'J0%'
ORDER BY f.casos_total DESC
LIMIT 10
"""

df_mspas_gold = query(SQL_MSPAS_GOLD)
print(f'Registros en Gold (MSPAS, J0x, depto Guatemala, 2020): {len(df_mspas_gold)}')
dark_df(df_mspas_gold)

---
## Resumen: ¿qué transforma cada capa?

In [ ]:
resumen = [
    {
        'Capa':        'BRONZE',
        'Propósito':   'Ingestión — copia fiel de la fuente',
        'Qué hace':    'Carga los archivos sin modificar + añade columnas de linaje (ingestion_timestamp, source_system, source_file)',
        'Limitaciones':'Tipos incorrectos · Códigos con separadores · Textos en inglés · Subtotales mezclados · Columnas de casos duplicadas',
    },
    {
        'Capa':        'SILVER',
        'Propósito':   'Limpieza y normalización',
        'Qué hace':    'Castea tipos (anio→INT, total→INT) · Normaliza CIE-10 (J:18:0 → J180) · Traduce etiquetas al español · Filtra subtotales · Resuelve departamentos contra catálogo · COALESCE de columnas de casos · Parsea grupos etarios (Age25_29 → 25-29) · Elimina duplicados exactos',
        'Limitaciones':'Sin modelo dimensional — tablas planas, sin surrogate keys, sin métricas consolidadas',
    },
    {
        'Capa':        'GOLD',
        'Propósito':   'Modelo dimensional — listo para análisis',
        'Qué hace':    'Galaxy Schema: 5 dimensiones compartidas + 2 tablas de hechos · Surrogate keys con row_number() · Periodo COVID calculado (anio >= 2020) · Unifica todas las fuentes en el mismo modelo · Deduplica por grain',
        'Limitaciones':'Las fuentes INE-edad e INE-geografía pueden contar las mismas muertes (filtrar por dim_fuente)',
    },
]

df_resumen = pd.DataFrame(resumen).set_index('Capa')

df_resumen.style.set_properties(**{
    'background-color': '#161b22',
    'color': '#c9d1d9',
    'font-size': '12px',
    'padding': '10px 14px',
    'text-align': 'left',
    'vertical-align': 'top',
    'white-space': 'pre-wrap',
    'border-color': '#30363d',
}).set_table_styles([
    {'selector': 'thead th', 'props': [
        ('background-color', '#21262d'), ('color', '#79c0ff'),
        ('padding', '10px 14px'), ('border-bottom', '2px solid #30363d'), ('font-size', '13px'),
    ]},
    {'selector': 'tbody tr:nth-child(1) th', 'props': [
        ('background-color', '#3d2012'), ('color', '#ffa657'), ('font-weight', '700'),
    ]},
    {'selector': 'tbody tr:nth-child(1) td', 'props': [
        ('background-color', '#1c1410'), ('color', '#ffa657'),
    ]},
    {'selector': 'tbody tr:nth-child(2) th', 'props': [
        ('background-color', '#112233'), ('color', '#79c0ff'), ('font-weight', '700'),
    ]},
    {'selector': 'tbody tr:nth-child(2) td', 'props': [
        ('background-color', '#0d1a24'), ('color', '#79c0ff'),
    ]},
    {'selector': 'tbody tr:nth-child(3) th', 'props': [
        ('background-color', '#2d1f00'), ('color', '#f0c040'), ('font-weight', '700'),
    ]},
    {'selector': 'tbody tr:nth-child(3) td', 'props': [
        ('background-color', '#1a1200'), ('color', '#e3b341'),
    ]},
    {'selector': 'table', 'props': [
        ('border', '1px solid #30363d'), ('border-radius', '8px'),
        ('overflow', 'hidden'), ('width', '100%'),
    ]},
])

In [11]:
conn.close()
print('Conexión cerrada — demo de trazabilidad ETL completo.')

Conexión cerrada — demo de trazabilidad ETL completo.
